# 05. Error analysis

Out-of-fold predictions of the selected configuration, **E9b**: 0.5 x fine-tuned e5 head + 0.5 x TF-IDF logistic regression (B2), with cross-fitted thresholds and both decoding rules, exactly as evaluated in CV. All 5,703 rows are CV rows; the test set is not used here.

All logic lives in `src/intent` (`models.error_analysis_data`, `evaluate.*`); this notebook calls it and narrates.

In [1]:
import pandas as pd

from intent.data import load_config, resolve
from intent.evaluate import (
    categorize_errors, category_examples, compare_systems, confused_pairs, conflicting_duplicate_mask,
    duplicate_crosscheck, error_category_counts, exact_match_ratio, export_manual_audit, find_label_errors,
    summary_metrics, temp_readings, top_label_errors, audit_agreement, audit_distribution,
)
from intent.models import error_analysis_data

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

cfg = load_config()
d = error_analysis_data(cfg)
labels, y, rows = d["labels"], d["y"], d["rows"].reset_index(drop=True)
pred, score = d["pred"]["E9b"], d["score"]["E9b"]
print(f"{len(rows)} CV rows, selected model E9b (out-of-fold)")

5703 CV rows, selected model E9b (out-of-fold)


Exact match ratio (subset accuracy: the whole label set must be right) next to macro F1. It is much stricter: a row with one extra or one missing label counts as fully wrong, which is why it sits well below the F1 scores.

In [2]:
pd.Series({
    "macro F1": summary_metrics(y.astype(bool), pred.astype(bool))["macro_f1"],
    "exact match ratio": exact_match_ratio(y, pred),
    "rows exactly right": int((y == pred).all(axis=1).sum()),
    "rows": len(y),
}).round(4)

macro F1                 0.7152
exact match ratio        0.6379
rows exactly right    3638.0000
rows                  5703.0000
dtype: float64

## 1. Likely label errors (cleanlab, multi-label mode)

cleanlab compares each given label set with the out-of-fold probabilities of a model that never saw the row. *suggested* is E9b's decoded prediction, i.e. a label set we would actually output.

In [3]:
issues, quality = find_label_errors(y, score)
conflicting = conflicting_duplicate_mask(rows["group_key"].to_numpy(), y)
print(f"cleanlab flags {issues.sum()} of {len(y)} rows ({issues.mean():.1%})")
top_label_errors(rows["text"], y, pred, quality, issues, conflicting, labels, k=10)

cleanlab flags 1413 of 5703 rows (24.8%)


,label_quality,text,given,suggested,conflicting_duplicate
2049,0.043,Desarrollo de nuevos proyectos por crecimiento requiero capital de trabajo,inv + renta,crec,False
103,0.052,Mejora de ubicación y mobiliario pago se servicios y posible ampliación,crec + equ,renta,False
2410,0.056,Consolidación de pasivos y capital de trabajo,equ + inv,cred,False
2440,0.057,PARA COMPRAR MÁS MATERIA PRIMA Y PODER SURTIR A MÁS CLIENTES,equ,inv,False
2551,0.057,Como capital de trabajo para crecimiento operario del negocio.,equ,crec + inv,False
2419,0.060,En compra de mercancías y estanteria y accesorios,equ,inv,False
2595,0.060,tengo un pequeño despacho creativo y quiero comprar equipo de computo y de mas herramientas así como también tomar c...,crec + renta,equ,False
3279,0.068,Quiero ampliar mi negocio para que reditue y seguir creciendo,renta,crec,False
1217,0.069,en ampliación de instalaciones por la alta demanda de servicio.,temp,renta,False
1146,0.070,Adquisición de equipo para venta importante,temp,equ,False


cleanlab flags about a quarter of the rows. Read that as an upper bound on noise, not an estimate: in multi-label mode a row is flagged if any of its 10 labels looks wrong, and the model's own systematic biases also raise flags.

The top of the ranking is convincing. Of the 10 rows, **7 look like annotation errors or incomplete labels** ("PARA COMPRAR MÁS MATERIA PRIMA" tagged equ instead of inv; "Consolidación de pasivos" tagged equ + inv instead of cred; "comprar equipo de cómputo" tagged crec + renta instead of equ; "mercancías y estantería" tagged equ only, missing inv), **1 is a model error** ("Mejora de ubicación y mobiliario ... posible ampliación": the given crec + equ is reasonable), and **2 are temp rows** whose text supports temp only loosely ("alta demanda", "venta importante").

### Cross-check with conflicting duplicates

Conflicting duplicates (same normalized text, different labels) are independent evidence of annotation noise. The canonical count is **58 groups (141 rows) on the clean data**, using the split's normalized key (ftfy, lowercase, collapsed whitespace). Exact string matching on the raw file gives 57, because it misses one pair that differs only by mojibake. This notebook works on the CV rows only, which hold 52 of the 58 groups (126 rows).

In [4]:
duplicate_crosscheck(issues, quality, conflicting, top=100)

,rows,flagged by cleanlab,in a conflicting duplicate group
all CV rows,5703,0.248,0.022
conflicting-duplicate rows,126,0.421,1.000
top 100 lowest label quality,100,1.000,0.060


Conflicting-duplicate rows are flagged far more often than average (42% vs 25%) and are about 2.7 times over-represented among the 100 lowest-quality rows (6.0% vs 2.2%). None reaches the top 10: one side of a conflicting pair is usually right, so these rows are suspicious but not extreme. cleanlab and the duplicates point to the same noise.

## 2. Error categories

An error is any row whose predicted label set differs from the given one. Each error is checked against every category below:

- **very short or vague**: at most 5 words, or every content token (after lemmatization and stopword removal) is generic: capital, trabajo, crecimiento, negocio, proyecto, liquidez, and similar.
- **multiple intents**, detected exactly as follows, with no parsing:
  1. the **given** set has 2 or more labels, **or**
  2. the **predicted** set has 2 or more labels **and** the lowercased, accent-stripped text contains a comma, " y ", " e ", "asi como" or "ademas".

  Rule 2 is a lexical proxy for "the text lists several things". It is deliberately loose: it also fires when the coordinated words belong to one use ("mover materiales y personal"), so this category is an upper bound. A one-item text with an extra predicted label does not count.
- **temp ambiguity** / **personal vs business**: the error involves the temp / no label (given and predicted disagree on it).
- **likely label error**: cleanlab flags the row (section 1).

The table shows two counts per category:
- **any applicable** counts every error the category fits. It does not depend on any ordering, but categories overlap, so the column sums to more than the number of errors.
- **primary cause** assigns each error to exactly one category: the first that fits in the order of the table. These counts sum to the number of errors, but **the primary shares depend on that order**: moving a category earlier can only increase its primary count.

Why this order: causes that explain the error from the text alone come first, from the least information (vague) to the most specific (temp, no); label noise comes last, as the residual explanation when the text is clear and single-intent but the model confidently disagrees. Putting cleanlab first would make it absorb most errors almost by definition, because cleanlab flags exactly the rows where the model disagrees with the label: 63% of all errors are flagged.

In [5]:
errors = categorize_errors(rows["text"], rows["text_tfidf"], y, pred, issues, labels, score)
print(f"{len(errors)} of {len(y)} rows ({len(errors) / len(y):.1%}) have at least one wrong label")
error_category_counts(errors)

2065 of 5703 rows (36.2%) have at least one wrong label


,primary cause,primary share,any applicable,any applicable share
very short or vague,54,0.026,54,0.026
multiple intents,1167,0.565,1181,0.572
temp ambiguity,96,0.046,201,0.097
personal vs business,126,0.061,160,0.077
likely label error,443,0.215,1303,0.631
other,179,0.087,179,0.087


The two columns tell different stories for the same data: likely label error is the primary cause of 21% of errors but applies to 63%; multiple intents applies to 57% and is primary for almost all of them because it is checked second.

Two examples per category, the ones where the model disagrees most confidently (repeated texts dropped):

In [6]:
category_examples(errors, k=2)

,primary,text,given,predicted
827,very short or vague,equipos de computo e impresoras,equ + inv,equ
502,very short or vague,Publicidad mejoras al local capacitación,mkt,mkt + renta
2625,multiple intents,para poder tener un fondo con que pagar proveedores y deudas cuando mis clientes no paguen a tiempo.,inv,cred + inv
178,multiple intents,Compra de mobiliario y equipo para la ampliación de Call Center.,equ,equ + renta
1290,temp ambiguity,Para dar credito a nuestros clientes en sus proyectos ya que ellos no pagas de 60 a 120 dias,cred,cred + temp
1146,temp ambiguity,Adquisición de equipo para venta importante,temp,equ
4551,personal vs business,"EN PAGAR LOS TRAMITES POR LA CONSTITUCION DE MI EMPRESA, EL REGISTRO DE MARCA, EL DISEÑO DE LA PÁGINA WEB Y EL DESAR...",no,mkt
1715,personal vs business,Para pagar gasolina y cosas que necesito en personales,equ,no
1923,likely label error,Comprar más inventarios.Comprar más herramientas Publicidad Cambiar imagen si es posible,inv,inv + mkt
3622,likely label error,para surtir más para mi tienda. TERMINAR UN LOCAL QUE ESTOY CONSTRUYENDO,inv,inv + renta


- **Very short or vague**: four-word lists leave the boundary open (printers: equ only, or equ + inv for consumables?); "Publicidad mejoras al local capacitación" is three nouns, and the model adds renta for "local".
- **Multiple intents**: the model tags a second stated use the annotator left out (debts in "pagar proveedores y deudas", premises in "ampliación de Call Center"). The first text also describes a payment-cycle gap, a temp reading nobody tagged.
- **Temp ambiguity**: "dar crédito a nuestros clientes ... no pagan de 60 a 120 días" is the receivables-cycle reading of temp, tagged cred only; "equipo para venta importante" is tagged temp with no timing in the text.
- **Personal vs business**: both examples look like label errors: incorporating a company and building its website tagged no; "gasolina y cosas que necesito en personales" tagged equ.
- **Likely label error**: incomplete labels: "Comprar más inventarios. Comprar más herramientas. Publicidad" tagged inv only (equ and mkt missing).
- **Other**: three or more uses in one sentence, where model and annotator each pick a different subset.

### Manual audit sample

50 errors drawn at random (seed 42) for hand labeling, to check how well the automatic categories match human judgment. Written to `reports/manual_audit.csv` (not versioned: it contains applicant text) with a `manual_category` column for the hand label. The file is never overwritten once any `manual_category` cell is filled, so re-running this notebook keeps the hand labels.

In [7]:
audit = export_manual_audit(errors, rows, resolve(cfg["paths"]["manual_audit"]), n=50, seed=cfg["seed"])
print(audit["assigned_category"].value_counts().to_string())
audit.head()

assigned_category
multiple intents        31
likely label error       6
personal vs business     5
other                    4
temp ambiguity           4


,source_row,record_pos,text,true_labels,predicted_labels,assigned_category,manual_category
0,3910,0,es un negocio de productos nutricionales la compañía se llama shaklee,crec,inic,other,model_error
1,6227,0,Invertir en estanterias y mercancia y remodelacion del local,renta,inv + renta,multiple intents,multiple intents
2,3916,0,mi negocio se dedica a la renta de luz y sonido,no,inic,personal vs business,likely label error
3,3886,0,EN MATERIAL Y ACTUALIZACION DE IMPRENTAS PARA MEJORA DE INFRAESTRUCTURA,inv,inv + renta,multiple intents,multiple intents
4,74,0,Promover pólizas de servicios funerarios con publicidad y promoción tanto en la iniciativa privada como en el sector...,inic,mkt,likely label error,likely label error


### Manual audit vs heuristic

All 50 rows were labeled by hand by one annotator. The instructions defined **other** as "text clear, label correct, model simply wrong"; the hand labels call that case **model_error**, so the two are treated as the same category (the strict, unmapped agreement is reported too). No row was labeled very short or vague.

In [8]:
agreement = audit_agreement(audit)
print(f"n = {agreement['n']}: agreement {agreement['agreement']:.0%} "
      f"(strict, before mapping model_error to other: {agreement['strict_agreement']:.0%}), "
      f"Cohen's kappa {agreement['kappa']:.2f}")
agreement["confusion"]

n = 50: agreement 80% (strict, before mapping model_error to other: 76%), Cohen's kappa 0.67


manual,multiple intents,temp ambiguity,personal vs business,likely label error,other
heuristic,,,,,
multiple intents,28,0,0,1,2
temp ambiguity,0,2,0,2,0
personal vs business,0,0,4,1,0
likely label error,0,0,0,4,2
other,1,0,0,1,2


Agreement is 80% (40 of 50) and Cohen's kappa is 0.67, substantial on the Landis and Koch scale; with 50 rows the kappa estimate is imprecise. The 10 disagreements are informative:

- **Model errors are invisible to the heuristics**: of the 6 rows judged model errors, only 2 landed in other; 2 were called likely label error and 2 multiple intents. cleanlab flags any confident disagreement between model and label, so it cannot tell a wrong model from a wrong label.
- **Half of the heuristic temp-ambiguity rows (2 of 4) were judged label errors**: the temp label itself is not supported by the text.
- **The likely label error flag is useful but imprecise**: 4 of the 6 rows it assigned were confirmed by hand.
- The rest are single cases: one multiple intents, one personal vs business and one other judged label errors, and one other judged multiple intents.

In [9]:
audit_distribution(audit, errors)

,manual n,manual share,wilson low,wilson high,heuristic share (same 50),heuristic primary share (all errors)
very short or vague,0,0.00,0.000,0.071,0.00,0.026
multiple intents,29,0.58,0.442,0.706,0.62,0.565
temp ambiguity,2,0.04,0.011,0.135,0.08,0.046
personal vs business,4,0.08,0.032,0.188,0.10,0.061
likely label error,9,0.18,0.098,0.308,0.12,0.215
other (manual: model_error),6,0.12,0.056,0.238,0.08,0.087


Every heuristic primary share over all 2,065 errors lies inside the 95% Wilson interval of the hand-labeled share. Multiple intents: 58% by hand [44%, 71%] vs 56.5% heuristic. Likely label error: 18% [10%, 31%] vs 21.5%.

**Model errors are 12% of errors by hand (6 of 50; 95% Wilson interval 5.6% to 23.8%)**, against 8.7% for the heuristic other. Scaled to the 2,065 CV errors, that is roughly 250 errors (between about 115 and 490) where the text is clear, the label is right and the model is wrong: the part of the error a better model could still fix. The rest is labeling convention or label noise.

**Check against the findings below: none is contradicted.** Finding 1 (multiple intents about 57%) matches the hand share of 58%. Finding 2 (real label noise) is supported: 18% of errors are label errors by hand. Finding 4 (temp is ill-defined) is reinforced: half of the heuristic temp errors are wrong temp labels. Finding 6 (personal vs business errors include label errors) holds: 1 of the 5 was a label error. Findings 3 and 5 are not tested by the audit. The findings are therefore unchanged.

Caveats: one annotator, who saw the heuristic category while labeling, so anchoring may inflate agreement; and 50 rows give wide intervals.

### How temp reads

Season words (temporada, navidad, diciembre, escolar, ...) vs receivables or payment-cycle words (cobro, factura, contrato, pedido, 30 to 120 días, liquidez, ...):

In [10]:
temp_readings(rows["text"], y, labels)

,temp rows,non-temp rows,share of temp rows
season,30,109,0.186
cycle,59,343,0.366
both,4,4,0.025
neither,68,5086,0.422


Of the 161 temp rows, 21% mention a season, 39% a payment cycle and 42% neither, while 113 non-temp rows mention a season and 347 mention cycle words. temp is two concepts plus an unexplained remainder, which caps any model on this label.

## 3. Most confused label pairs

Within the same row: a given label the model missed, and a label it predicted instead. *share* is relative to the missed label's support.

In [11]:
confused_pairs(y, pred, labels, top=10)

,missed (given),predicted instead,rows,share_of_missed_label
0,equ,inv,102,0.060
1,inv,equ,53,0.027
2,inv,crec,49,0.025
3,crec,inv,43,0.084
4,renta,equ,31,0.047
5,crec,renta,31,0.060
6,crec,inic,28,0.054
7,renta,crec,28,0.043
8,temp,inv,28,0.174
9,equ,renta,25,0.015


equ vs inv dominates in both directions (102 + 53 rows): durable goods vs stock is the main boundary, and the codebook does not define inv. crec trades places with inv, renta and inic in both directions, because "growth" overlaps every specific use. By share, temp to inv is the worst pair: 17% of temp rows lose temp to inv (seasonal stock purchases).

## 4. Fine-tuned model (E4) vs TF-IDF (B2)

Per-row F1 of each system's decoded prediction. Examples are rows where the winner is exactly right and the loser was most confidently wrong.

In [12]:
summary, e4_wins, b2_wins = compare_systems(
    rows["text"], y, d["pred"]["E4"], d["pred"]["B2"], d["score"]["E4"], d["score"]["B2"], labels, k=5,
)
summary.rename({"a_better": "E4 better", "b_better": "B2 better", "same": "same row F1"})

E4 better      1000
B2 better       695
same row F1    4008
dtype: int64

In [13]:
e4_wins.rename(columns={"winner": "E4 (fine-tuned)", "loser": "B2"})

,text,given,E4 (fine-tuned),B2
2048,Invertir en medicamento y la compra de una copiadora y poder ampliar el local y en general surtir lo que es producto...,equ + renta,equ + renta,inv
405,Un automóvil para poder mover materiales y personal,equ,equ,inv + sueldo
2792,comprar un traje,no,no,equ + inv
2169,en invertir en tecnología para mi negocio,equ,equ,crec + inic
3506,"El dinero lo quiero para invertir en maquinaria, materia prima y pagar una parte de mis deudas y dejar algo de capit...",cred + equ,cred + equ,inv


Where the fine-tuned model wins:
1. *Medicamento, copiadora, ampliar el local, surtir papelería*: B2 keys on "surtir productos" and predicts inv only; E4 picks the copier (equ) and the premises (renta), matching the annotator, who did not tag the restocking.
2. *Un automóvil para mover materiales y personal*: B2 fires on the words "materiales" (inv) and "personal" (sueldo); E4 sees that the purchase is a vehicle.
3. *Comprar un traje*: "comprar" pulls B2 toward business purchases; E4 recognizes personal consumption.
4. *Invertir en tecnología para mi negocio*: B2 reads "invertir en mi negocio" as growth or start-up; E4 maps "tecnología" to equipment.
5. *Maquinaria, materia prima y pagar deudas*: E4 recovers both annotated uses; B2 keeps only "materia prima" (inv), which the annotator left untagged, another incomplete multi-use label.

In [14]:
b2_wins.rename(columns={"winner": "B2", "loser": "E4 (fine-tuned)"})

,text,given,B2,E4 (fine-tuned)
670,requiero una linea de crédito por que vendo a una cadena nacional y me paga a 90 días y de momento estoy pagando una...,cred,cred,crec + inv + temp
4577,INCREMENTAR EL CAPITAL DE TRABAJO Y CONSEGUIR MEJORES PRECIOS AL PAGAR DE CONTADO,inv,inv,crec + renta
3595,soy consultora de belleza de mary kay cosmetics. y quiero el dinero para asi poder impulsar mi negocio..,inv,inv,crec + inic
3840,invertir en mi negocio que es una tienda de abarrotes,inv,inv,crec + inic
2854,yo me didico a vender ropa y la llevo al puerto de tuxpan veracruz y necesito para invertir mas ya q es una plaza q ...,inv,inv,crec + inic


Where TF-IDF wins:
1. *Línea de crédito, la cadena paga a 90 días, factoraje, ampliar la planta*: E4 tags the 90-day cycle as temp and the plant as crec, arguably faithful to the text; the annotator coded only the credit line. A codebook problem more than a model error.
2. *Incrementar el capital de trabajo y pagar de contado para mejores precios*: buying stock in cash is inv; B2 learned that phrasing, E4 reads "incrementar el capital" as growth (and adds an unsupported renta).
3. *Consultora de Mary Kay, impulsar mi negocio*: the annotator inferred inv from the business type (a cosmetics reseller); B2 has learned such cues, E4 reads "impulsar mi negocio" literally as growth or start-up.
4. *Invertir en mi tienda de abarrotes*: same pattern; a grocery store implies stock purchases to the annotator and to B2, but the text only says "invest in my business".
5. *Vendo ropa en Tuxpan, necesito invertir más*: a clothing reseller is inv by annotator convention; E4 predicts growth.

The two systems fail differently: E4 reads what is being bought, B2 carries the annotators' business-type priors. That complementarity is why the blend beats either model alone.

## 5. Codebook recommendations

- **Define inv**: goods bought for resale, raw materials and production inputs (mercancía, materia prima, insumos). Durable assets used for more than a year (machinery, tools, vehicles, furniture, computers) are equ. Also decide whether annotators may infer inv from the business type ("tienda de abarrotes") when no use is stated; today they do, inconsistently.
- **Split or redefine temp**: split into *temp_season* (stock or staff for a demand peak) and *temp_cycle* (financing the gap between paying suppliers and collecting from clients: contracts, invoices at 30 to 120 days, factoring). If one label must stay, define it as working capital that bridges a timing gap and require the text to state that timing.
- **crec as the residual**: tag crec only when no specific use is named ("crecer", "capital de trabajo para crecimiento"); when the text names a use, tag that use instead.
- **Multi-use rule**: tag every stated use, not only the main one.

## Findings

1. **Most errors are a labeling-convention problem**: 1,181 of 2,065 errors (57%) involve a multi-use text where model and annotator tag different subsets of the stated uses (1,167 as primary cause; the share barely depends on rule order). Only 63.8% of rows get the exact label set right, against a macro F1 of 0.715.
2. **Label noise is real and locatable**: 7 of cleanlab's top 10 are clear annotation errors, and conflicting duplicates are flagged at 42% vs 25% overall.
3. **equ vs inv is the largest confusion** (155 rows across both directions), a direct consequence of inv being undefined in the codebook.
4. **temp mixes two concepts**: 21% of temp rows mention a season, 39% a payment cycle, 42% neither, and 17% of temp rows are predicted as inv; this explains temp's F1 ceiling near 0.4.
5. **The fine-tuned model and TF-IDF fail differently** (E4 better on 1,000 rows, B2 on 695): E4 reads the purchase object, B2 carries business-type priors, which is why the blend wins.
6. **Personal vs business errors cluster on vehicles, housing and self-employment**, and several are label errors (a company's incorporation tagged no).